In [1]:
import torch
import deepinv
from torchvision import datasets, transforms

/Users/ishittaiyer/Desktop/Research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import certifi

os.environ['SSL_CERT_FILE'] = certifi.where()

In [3]:
device = "cpu"
image_size = 32
batch_size = 16

In [4]:
# preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.0,), (1.0,)),
])

In [5]:
# loading MNIST
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root="./data", train=True, download=True, transform=transform),
    batch_size = batch_size,
    shuffle = True
)

In [6]:
lr = 1e-4
epochs = 3

In [7]:
# using DiffUNet
model = deepinv.models.DiffUNet(in_channels=1, out_channels=1, pretrained=None).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
mse = deepinv.loss.MSE(reduction='mean')

In [10]:
# beta schedule -> linear like in the paper
beta_start = 1e-4
beta_end = 0.02
timesteps = 100
betas = torch.linspace(beta_start, beta_end, timesteps, device=device)

# alphas and their compliments
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

for epoch in range(epochs):
    model.train()
    for i, (data, _) in enumerate(train_loader):
        imgs = data.to(device)
        noise = torch.randn_like(imgs)
        t = torch.randint(0, timesteps, (imgs.size(0),), device=device)
        
        noised_imgs = (
            sqrt_alphas_cumprod[t, None, None, None] * imgs
            + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
        )

        optimizer.zero_grad()
        estimated_noise = model(noised_imgs, t, type_t="timestep")
        loss = mse(estimated_noise, noise).mean()
        loss.backward()
        optimizer.step()
        if i % 50 == 0:
            print(f"epoch {epoch} iter {i}/{len(train_loader)} loss {loss.item():.4f}")


torch.save(
    model.state_dict(),
    "trained_diffusion_model.pth",
    )
        

epoch 0 iter 0/3750 loss 0.5088
epoch 0 iter 50/3750 loss 0.0841
epoch 0 iter 100/3750 loss 0.0559
epoch 0 iter 150/3750 loss 0.0385
epoch 0 iter 200/3750 loss 0.0358
epoch 0 iter 250/3750 loss 0.0652
epoch 0 iter 300/3750 loss 0.0690
epoch 0 iter 350/3750 loss 0.0404
epoch 0 iter 400/3750 loss 0.0429
epoch 0 iter 450/3750 loss 0.0364
epoch 0 iter 500/3750 loss 0.0293


KeyboardInterrupt: 